# LLM Evaluation with LangSmith
Revision notebook: create datasets, run a target application, use LLM-as-a-judge for correctness and concision, and log experiments in LangSmith.

**Workflow:** Dataset → Target run → Evaluators → Experiment results in LangSmith.

SDK APIs evolve, so if a cell differs from your installed version, check the official docs linked at the end. Use environment variables for secrets.

## 1. Install dependencies
Install the LangSmith SDK and the LangChain provider used for the target/judge model.

In [ ]:
%pip install -U langsmith langchain-core langchain-openai

## 2. Configure credentials
Set `LANGSMITH_API_KEY` and `OPENAI_API_KEY` in your shell or notebook secret manager. This cell prompts if they are missing.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API key: ")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "llm-eval-revision")

## 3. Dataset concepts

A LangSmith dataset contains examples:
- `inputs`: what your application receives,
- `outputs`: reference/expected answers, when available,
- `metadata`: optional labels such as topic or difficulty.

A useful evaluation set represents real use, including edge cases and known failures—not just easy examples. Review references for quality and remove sensitive data before uploading.

In [ ]:
examples = [
    {
        "inputs": {"question": "What is the capital of France?"},
        "outputs": {"answer": "Paris."},
        "metadata": {"topic": "geography", "difficulty": "easy"},
    },
    {
        "inputs": {"question": "What does CPU stand for?"},
        "outputs": {"answer": "Central Processing Unit."},
        "metadata": {"topic": "computing", "difficulty": "easy"},
    },
    {
        "inputs": {"question": "Why does correlation not prove causation?"},
        "outputs": {"answer": (
            "A third variable may influence both, causality may run in the opposite "
            "direction, or the association may be coincidental. Stronger causal "
            "evidence or a suitable study design is needed."
        )},
        "metadata": {"topic": "reasoning", "difficulty": "medium"},
    },
]

## 4. Create and populate a LangSmith dataset
This creates a new dataset and uploads the examples. If you rerun it, use a new versioned name or look up the existing dataset to avoid duplicate-name errors.

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "llm-eval-correctness-concision-v1"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="QA examples for evaluating answer correctness and concision.",
)
client.create_examples(dataset_id=dataset.id, examples=examples)

print("Dataset:", dataset.name)
print("Dataset ID:", dataset.id)

## 5. Inspect the dataset
Verify inputs, references, and metadata before spending model calls on evaluation.

In [ ]:
dataset_examples = list(client.list_examples(dataset_id=dataset.id))
print("Example count:", len(dataset_examples))
for example in dataset_examples:
    print("INPUT:", example.inputs)
    print("REFERENCE:", example.outputs)
    print("METADATA:", example.metadata)
    print("-" * 50)

## 6. Define the target application

The target is the application being evaluated. This demo uses a simple model call; replace it with your RAG chain or agent later. The function receives an input dictionary and returns an output dictionary.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

target_model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def target(inputs: dict) -> dict:
    response = target_model.invoke([
        SystemMessage(content=(
            "Answer accurately and concisely, usually in one to three sentences. "
            "If uncertain, say so instead of inventing facts."
        )),
        HumanMessage(content=inputs["question"]),
    ])
    return {"answer": response.content}

## 7. LLM-as-a-judge: rubric design

A judge model applies an explicit rubric to a candidate answer and returns a score plus rationale.

**Correctness** checks factual accuracy, key information, and unsupported claims against the question and reference.

**Concision** checks whether the answer is focused and appropriately brief without omitting information needed to answer.

Use a stable 1–5 scale: 1 = very poor, 2 = substantial issues, 3 = mixed, 4 = good, 5 = excellent. Judges are imperfect; calibrate them against human-reviewed examples.

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class JudgeResult(BaseModel):
    score: int = Field(ge=1, le=5)
    rationale: str

judge_model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
).with_structured_output(JudgeResult)

## 8. Correctness evaluator
LangSmith evaluator functions can receive `inputs`, `outputs`, and `reference_outputs`. Return a score and a comment so results are interpretable.

In [ ]:
def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    question = inputs.get("question", "")
    candidate = outputs.get("answer", "")
    reference = reference_outputs.get("answer", "")

    prompt = f'''
You are a strict but fair evaluator of answer correctness.

Question: {question}
Reference answer: {reference}
Candidate answer: {candidate}

Score from 1 to 5:
5 = factually correct and complete on essential facts
4 = correct with only a minor omission or imprecision
3 = partially correct or meaningfully incomplete
2 = major error or serious omission
1 = mostly wrong, fabricated, or non-responsive

Accept semantically equivalent wording. Penalize unsupported claims.
Return a score and a brief rationale.
'''
    judged = judge_model.invoke(prompt)
    return {"key": "correctness", "score": judged.score, "comment": judged.rationale}

## 9. Concision evaluator
Do not reward an answer merely for being short: it must still include the necessary information.

In [ ]:
def concision_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    question = inputs.get("question", "")
    candidate = outputs.get("answer", "")
    reference = reference_outputs.get("answer", "")

    prompt = f'''
Evaluate answer concision while preserving usefulness.

Question: {question}
Reference answer (for expected scope, not exact wording): {reference}
Candidate answer: {candidate}

Score from 1 to 5:
5 = concise, focused, and includes needed information
4 = mostly concise, with minor unnecessary detail
3 = noticeably wordy or slightly under-explained
2 = substantially verbose/repetitive or too terse to be useful
1 = extremely rambling or so short it fails to answer

Do not reward brevity that sacrifices correctness or essential context.
Return a score and a brief rationale.
'''
    judged = judge_model.invoke(prompt)
    return {"key": "concision", "score": judged.score, "comment": judged.rationale}

## 10. Run evaluation and log the experiment

`evaluate(...)` runs the target on each dataset example and logs the experiment and evaluator results to LangSmith. Inspect individual examples, outputs, scores, and rationales in the LangSmith UI.

In [ ]:
from langsmith import evaluate

results = evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness_evaluator, concision_evaluator],
    experiment_prefix="qa-correctness-concision",
    description="Baseline target evaluated for correctness and concision.",
)
print(results)

## 11. Add a deterministic metric

Use deterministic metrics for objective properties, and judge metrics for semantic properties. Word count is a useful supporting signal, but it is not a complete definition of concision.

In [ ]:
def word_count_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    answer = outputs.get("answer", "")
    count = len(answer.split())
    return {
        "key": "under_80_words",
        "score": 1 if count <= 80 else 0,
        "comment": f"Answer contains {count} whitespace-separated words.",
    }

# To include it in a new experiment:
# results = evaluate(
#     target,
#     data=dataset_name,
#     evaluators=[correctness_evaluator, concision_evaluator, word_count_evaluator],
#     experiment_prefix="qa-three-metrics",
# )

## 12. Improve the evaluation set

Add examples for:
- common and rare user intents,
- ambiguous or underspecified questions,
- questions where uncertainty is the correct response,
- long answers that are necessary,
- cases where a short answer is sufficient,
- known production failures (with sensitive data removed).

Have humans score a representative subset. Compare human scores with judge scores, inspect disagreements, and refine the rubric.

## 13. Common mistakes and production checklist

- Vague judge rubrics produce inconsistent scores.
- Correctness needs a reference or explicit criteria.
- Concision must not reward missing information.
- Toy datasets can make a weak system look strong.
- Inspect individual failures, not only average scores.
- Version datasets and experiment prefixes meaningfully.
- Consider judge bias and calibrate against human ratings.
- Keep secrets and sensitive customer data out of traces unless approved.
- Rerun the same dataset after changing one variable so comparisons are meaningful.

### Quick revision
- **Dataset:** examples with inputs, references, metadata.
- **Target:** the application under test.
- **Evaluator:** function that scores a run.
- **LLM-as-judge:** model applies a written rubric.
- **Experiment:** target runs plus evaluation results logged in LangSmith.

## Official references

- [LangSmith evaluation overview](https://docs.smith.langchain.com/evaluation)
- [Manage datasets](https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets)
- [LLM-as-a-judge guide](https://docs.smith.langchain.com/evaluation/how_to_guides/llm_as_judge)
- [LangSmith Python SDK](https://docs.smith.langchain.com/reference/python)

**Version note:** if `evaluate` or evaluator signatures differ in your environment, follow the current docs for the installed LangSmith SDK.